In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Importing libraries
import warnings
warnings.filterwarnings('ignore')
import torch
import numpy as np
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
print("it is done")

it is done


In [4]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device=="cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

Device: cuda
GPU name: Tesla T4


In [5]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import os

# ===============================
# Device
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ===============================
# Transforms (Medical Image Friendly)
# ===============================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ===============================
# Dataset Path (Colab)
# ===============================
DATASET_PATH = "/content/drive/Othercomputers/الكمبيوتر المحمول/Desktop/data_tyumor/data_tyumor/Dataset/Brain Tumor MRI images"
assert os.path.exists(DATASET_PATH), "Dataset path not found"

# ===============================
# Load Dataset
# ===============================
full_dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=train_transform
)

num_classes = len(full_dataset.classes)
print("Classes:", full_dataset.classes)

# ===============================
# Stratified-like Split (Stable)
# ===============================
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, test_size],
    generator=generator
)

# Apply correct transform to test set
test_dataset.dataset.transform = test_transform

# ===============================
# DataLoaders (Optimized for GPU)
# ===============================
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")


Using device: cuda
Classes: ['Healthy', 'Tumor']
Train samples: 2907
Test samples: 727


In [6]:

def show_MRI_batch(dataloader, title="Batch of mri Images"):
    images, labels = next(iter(dataloader))
    fig, axes = plt.subplots(4, 8, figsize=(15, 8))
    fig.suptitle(title)

    for i, ax in enumerate(axes.flatten()):
        if i < len(images):
            img = images[i].permute(1, 2, 0)  # Convert tensor image for plotting
            ax.imshow(img)
            ax.set_title(MRI.classes[labels[i]])
            ax.axis('off')
    plt.show()

print("it is done")

it is done


In [7]:
import torch
import torch.nn as nn

# ===============================
# Conv Block
# ===============================
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.block(x)


# ===============================
# MRI CNN Model (Improved)
# ===============================
class MRIModel(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),    # 224 → 112
            ConvBlock(32, 64),             # 112 → 56
            ConvBlock(64, 128),            # 56 → 28
            ConvBlock(128, 256),           # 28 → 14
            ConvBlock(256, 512)            # 14 → 7
        )

        # Global Average Pooling بدل Flatten
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),       # 7x7 → 1x1
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ===============================
# Instantiate Model
# ===============================
mri_model = MRIModel(in_channels=3, num_classes=2)
print(mri_model)


MRIModel(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): ConvBlock(
      (block): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(mri_model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

scaler = torch.cuda.amp.GradScaler()

mri_model = mri_model.to(device)


In [ ]:
num_epochs = 30
best_f1 = 0.0
patience = 5
counter = 0
for epoch in range(num_epochs):

    # ======================
    # Training
    # ======================
    mri_model.train()
    train_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} Training"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = mri_model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(mri_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ======================
    # Validation
    # ======================
    mri_model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Epoch {epoch+1} Validation"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = mri_model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_f1 = f1_score(all_labels, all_preds, average='weighted')
    scheduler.step(val_f1)

    # ======================
    # Save Best Model
    # ======================
    if val_f1 > best_f1:
        best_f1 = val_f1
        counter = 0
        torch.save(mri_model.state_dict(), "best_mri_model.pth")
    else:
        counter += 1

    # ======================
    # Early Stopping
    # ======================
    if counter >= patience:
        print("Early stopping triggered")
        break

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val F1-score: {val_f1:.4f}"
    )


Epoch 1 Validation: 100%|██████████| 29/29 [00:34<00:00,  1.17s/it]


Epoch [1/30] | Train Loss: 0.3222 | Val F1-score: 0.3381


Epoch 2 Validation: 100%|██████████| 29/29 [00:06<00:00,  4.81it/s]


Epoch [2/30] | Train Loss: 0.2576 | Val F1-score: 0.9501


Epoch 3 Validation: 100%|██████████| 29/29 [00:07<00:00,  4.02it/s]


...

Epoch [14/30] | Train Loss: 0.1126 | Val F1-score: 0.9794


Epoch 15 Validation: 100%|██████████| 29/29 [00:06<00:00,  4.31it/s]


Epoch [15/30] | Train Loss: 0.1032 | Val F1-score: 0.9675


Epoch 16 Validation: 100%|██████████| 29/29 [00:07<00:00,  3.77it/s]

Early stopping triggered


In [2]:
# import matplotlib.pyplot as plt
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# # ===========================
# # Compute confusion matrix
# # ===========================
# cm = confusion_matrix(, mri_all_predictions)

# # ===========================
# # Display
# # ===========================
# disp = ConfusionMatrixDisplay(
#     confusion_matrix=cm,
#     display_labels=["Healthy", "Tumor"]  # عدّل حسب الفئات الفعلية
# )

# fig, ax = plt.subplots(figsize=(6,6))
# disp.plot(ax=ax, cmap=plt.cm.Blues, colorbar=True)
# plt.title("Confusion Matrix")
# plt.show()

# # ===========================
# # Optional: save figure
# # ===========================
# fig.savefig("confusion_matrix.png", dpi=300, bbox_inches='tight')


In [ ]:
# حفظ أفضل نموذج بعد التدريب
torch.save(mri_model.state_dict(), "mri_model.pth")
print("Model saved successfully!")
# تعريف نفس هيكل النموذج
mri_loaded = MRIModel(in_channels=3, num_classes=2)
mri_loaded.load_state_dict(torch.load("mri_model.pth"))
mri_loaded = mri_loaded.to(device)
mri_loaded.eval()  # وضع النموذج في وضع التقييم
# ===============================
# Save model
# ===============================
torch.save(mri_model.state_dict(), "mri_model.pth")
print("Model saved!")

# ===============================
# Download to local machine
# ===============================
from google.colab import files
files.download("mri_model.pth")

# بعد تدريب النموذج
import torch

# لنفترض أن اسم النموذج بعد التدريب هو mri_model
MODEL_PATH = "mri_model_full.pth"

# حفظ النموذج كاملًا
torch.save(mri_model, MODEL_PATH)

print(f"Model saved successfully at {MODEL_PATH}")



Model saved successfully!
Model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model saved successfully at mri_model_full.pth


In [ ]:
from google.colab import files

# اسم النموذج الكامل الذي تريد تنزيله
model_path = "mri_model_full.pth"

# تنزيل النموذج مباشرة على جهازك
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>